# Category segmentation for articles


In [0]:
# COMMAND ---------- 
# Cell 1 – Install & basic setup
# (Run once after cluster start; may take 1-2 minutes)
%pip install --quiet --upgrade sentence-transformers scikit-learn umap-learn hdbscan==0.8.33
dbutils.library.restartPython()   # <- restarts the Python context so the new libs are visible

In [0]:
# COMMAND ----------
# Cell 2 – Imports & constants
from pyspark.sql import functions as F, types as T
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np, pandas as pd, re, unicodedata, string, pathlib

In [0]:
DATA_PATH         = "/FileStore/tables/stockcode_db.csv"
ENCODING          = "latin1"
SEP               = ";"        # the csv is ";"-separated
MIN_WORD_LEN      = 3          # drop very short tokens in cleaning
MODEL_NAME        = "all-MiniLM-L6-v2"
min_cluster_size  = 10         # clusters smaller than this -> H1_Other
conf_threshold    = 0.80       # probability < 0.80 -> H1_Other

# cluster granularity per level (tweakable)
n_clusters_lvl1, n_clusters_lvl2, n_clusters_lvl3 = 10, 30, 80   

In [0]:
# COMMAND ----------
# Cell 3 – Load csv into a Spark DF
raw_df = (
    spark.read
         .option("header", "true")
         .option("sep", SEP)
         .option("encoding", ENCODING)
         .csv(DATA_PATH)
         .select(
             F.trim(F.col("StockCode")).alias("StockCode"),
             F.trim(F.col("Description")).alias("Description")
         )
         .dropna(subset=["Description"])   # a handful of blanks
         .dropDuplicates(["StockCode"])     # one row per StockCode
)
display(raw_df.limit(10))

In [0]:
# COMMAND ----------
# Cell 4 – Text cleaning helper (UDF)
def basic_clean(txt: str) -> str:
    txt = txt.lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode()
    txt = re.sub(r"[{}]".format(re.escape(string.punctuation)), " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    txt = " ".join([w for w in txt.split() if len(w) >= MIN_WORD_LEN])
    return txt

clean_udf = F.udf(basic_clean, T.StringType())
clean_df = raw_df.withColumn("clean_desc", clean_udf("Description"))

In [0]:
# COMMAND ----------
# Cell 5 – Sentence-BERT embeddings (vector size 384)
model = SentenceTransformer(MODEL_NAME)
@F.pandas_udf("array<float>")
def embed_series(col: pd.Series) -> pd.Series:
    return pd.Series(model.encode(col.to_list(), show_progress_bar=True).tolist())

vect_df = clean_df.withColumn("emb", embed_series("clean_desc"))

In [0]:
# COMMAND ----------
# Cell 6 – Agglomerative clustering per hierarchy level
# collect embeddings in driver memory (fits easily in Free Edition)
vect_pd = vect_df.select("StockCode", "clean_desc", "emb").toPandas()
emb      = np.vstack(vect_pd["emb"].values)

def cluster_and_append(df_pd, emb_mat, n_clusters, col_name):
    clus = AgglomerativeClustering(n_clusters=n_clusters, metric="cosine", linkage="average")
    df_pd[col_name] = clus.fit_predict(emb_mat)
    return df_pd

vect_pd = cluster_and_append(vect_pd, emb, n_clusters_lvl1, "lvl1_id")
vect_pd = cluster_and_append(vect_pd, emb, n_clusters_lvl2, "lvl2_id")
vect_pd = cluster_and_append(vect_pd, emb, n_clusters_lvl3, "lvl3_id")

In [0]:
# COMMAND ----------
# Cell 7 – Human-readable labels from top-TF-IDF terms
def id_to_label(df, id_col, prefix="H1_"):
    tfidf = TfidfVectorizer(max_features=3_000)
    tfidf.fit(df["clean_desc"])
    labels = {}
    for cid, group in df.groupby(id_col):
        top_idx = tfidf.transform(group["clean_desc"]).sum(axis=0).A1.argsort()[-5:][::-1]
        words   = [tfidf.get_feature_names_out()[i] for i in top_idx if len(tfidf.get_feature_names_out()[i])>=MIN_WORD_LEN]
        label   = prefix + (" ".join(words[:3]).upper().replace(" ", "_") or "MISC")
        labels[cid] = label
    return df[id_col].map(labels)

vect_pd["lvl1_label"] = id_to_label(vect_pd, "lvl1_id")
vect_pd["lvl2_label"] = id_to_label(vect_pd, "lvl2_id")
vect_pd["lvl3_label"] = id_to_label(vect_pd, "lvl3_id")

In [0]:
# COMMAND ----------
# Cell 8 – Apply “H1_Other” fallback for tiny / low-confidence clusters
# (Confidence proxy =  cluster size / total size)
total = len(vect_pd)
lvl_other_mask = vect_pd.groupby("lvl3_id")["StockCode"].transform("count") < min_cluster_size
vect_pd.loc[lvl_other_mask, ["lvl1_label","lvl2_label","lvl3_label"]] = ["H1_OTHER"]*3

In [0]:
# COMMAND ----------
# Cell 9 – Create final mapping Spark DF
map_pd = vect_pd[["StockCode","lvl1_label","lvl2_label","lvl3_label"]]
map_spark = (
    spark.createDataFrame(map_pd)
          .withColumnRenamed("lvl1_label","H1_lvl1")
          .withColumnRenamed("lvl2_label","H1_lvl2")
          .withColumnRenamed("lvl3_label","H1_lvl3")
)

# Quick sanity checks
display(map_spark.groupBy("H1_lvl1").count().orderBy(F.desc("count")))
display(map_spark.filter(F.col("H1_lvl1")=="H1_OTHER").count())

In [0]:
# COMMAND ----------
# Cell 10 – Persist mapping for downstream joins
TARGET_TABLE = "stockcode_category_map"   # change if you prefer a path
(
  map_spark.write
           .format("delta")
           .mode("overwrite")
           .saveAsTable(TARGET_TABLE)
)
print(f"  Mapping table written to '{TARGET_TABLE}'")

In [0]:
# COMMAND ----------
# Cell 11 – (OPTIONAL) Export preview for manual QA
tmp_path = "/tmp/stockcode_mapping_preview.csv"
(map_spark.limit(2000)          # sample – edit as needed
           .toPandas()
           .to_csv(tmp_path, index=False)
)
dbutils.fs.cp(f"file:{tmp_path}", "dbfs:/FileStore/stockcode_mapping_preview.csv")
print("Preview saved to /FileStore/stockcode_mapping_preview.csv")